# DATA 605 - Week 4 Assignment
## Centrality Measures - Network Data Plan
Author: Tanvir Raju  
Date: May 2026

---
## 1. Dataset Selection

### Dataset: Political Blog Network (Adamic & Glance, 2005)

Source: Stanford Network Analysis Project (SNAP)  
URL: https://snap.stanford.edu/data/political-blogs.html

This dataset represents the US political blogosphere around the time of the 2004 presidential election. It was compiled by Adamic and Glance (2005) and has since become a benchmark dataset for community detection and centrality analysis in political networks.

### What the Dataset Contains

- Nodes: 1,490 political blog websites
- Edges: 19,090 hyperlinks between blogs (directed - one blog linking to another)
- Categorical variable: Political leaning of each blog
  - 0 = Liberal
  - 1 = Conservative

### Why This Dataset is a Good Choice

- Freely available, no API key or web scraping required
- Clean and well-documented with a clear categorical variable
- Large enough (1,490 nodes) to make centrality comparisons meaningful
- Politically relevant - centrality differences between Liberal and Conservative blogs have real-world interpretation
- Has been used in published research, so findings can be compared against known results
- Both directed and undirected versions can be analyzed, enabling comparison of in-degree vs out-degree centrality

---
## 2. Data Loading Plan

The dataset consists of two files:

| File | Contents |
|---|---|
| nodes.csv | Node ID, blog name, political leaning (0=Liberal, 1=Conservative) |
| edges.csv | Source node ID, Target node ID (hyperlink from source to target) |

### Step-by-Step Loading Plan

Step 1 - Import libraries
```python
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
```

Step 2 - Load node data
```python
nodes_df = pd.read_csv('nodes.csv')
nodes_df['leaning_label'] = nodes_df['leaning'].map({0: 'Liberal', 1: 'Conservative'})
```

Step 3 - Load edge data
```python
edges_df = pd.read_csv('edges.csv')
```

Step 4 - Build the directed graph
```python
G = nx.DiGraph()
G.add_nodes_from(nodes_df['node_id'])
nx.set_node_attributes(G, nodes_df.set_index('node_id')['leaning_label'].to_dict(), 'leaning')
G.add_edges_from(zip(edges_df['source'], edges_df['target']))
```

Step 5 - Compute centrality measures
```python
degree_centrality      = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
eigenvector_centrality = nx.eigenvector_centrality(G, max_iter=500)
in_degree_centrality   = nx.in_degree_centrality(G)
out_degree_centrality  = nx.out_degree_centrality(G)
```

Step 6 - Attach centrality scores back to nodes dataframe
```python
nodes_df['degree_centrality']      = nodes_df['node_id'].map(degree_centrality)
nodes_df['betweenness_centrality'] = nodes_df['node_id'].map(betweenness_centrality)
nodes_df['eigenvector_centrality'] = nodes_df['node_id'].map(eigenvector_centrality)
nodes_df['in_degree_centrality']   = nodes_df['node_id'].map(in_degree_centrality)
nodes_df['out_degree_centrality']  = nodes_df['node_id'].map(out_degree_centrality)
```

Step 7 - Compare centrality across categorical groups
```python
summary = nodes_df.groupby('leaning_label')[[
    'degree_centrality',
    'betweenness_centrality',
    'eigenvector_centrality'
]].mean()
print(summary)
```

Step 8 - Visualize the network colored by political leaning
```python
color_map = {'Liberal': '#1f77b4', 'Conservative': '#d62728'}
node_colors = [color_map[G.nodes[n]['leaning']] for n in G.nodes()]

plt.figure(figsize=(14, 10))
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos, node_color=node_colors, node_size=20,
                 with_labels=False, edge_color='gray', alpha=0.5, arrows=False)
plt.title('Political Blog Network - Liberal (blue) vs Conservative (red)')
plt.show()
```

---
## 3. Categorical Variable

The key categorical variable is political leaning, assigned to each blog node:

| Value | Label | Description |
|---|---|---|
| 0 | Liberal | Blog identified as left-leaning or Democrat-aligned |
| 1 | Conservative | Blog identified as right-leaning or Republican-aligned |

This binary categorical variable allows direct comparison of centrality distributions between the two political groups. It was manually labeled by Adamic and Glance based on blog content and self-identification.

Additional categorical variables that could be derived:
- Bridge vs. Silo: whether a blog links across political lines or only within its group
- High vs. Low influence: based on whether eigenvector centrality exceeds the median

---
## 4. Hypothetical Outcome

### Hypothesis

Conservative blogs will exhibit higher average degree centrality within the full network, while Liberal blogs will show higher betweenness centrality - indicating that Liberal blogs are more likely to serve as bridges between communities, while Conservative blogs form a more densely interconnected internal cluster.

### Reasoning

Research on political networks consistently finds that online political communities tend to cluster by ideology - a phenomenon known as echo chambers. Within such a structure, we would expect:

Degree Centrality:  
If Conservative blogs form a denser internal cluster (more links to each other), their average degree centrality within the full network would be higher simply because they have more mutual connections. A blog with many links to other blogs within a tight cluster accumulates a high degree count even if it never bridges across to the other side.

Betweenness Centrality:  
Betweenness centrality measures how often a node sits on the shortest path between two other nodes. If Liberal blogs are more likely to link across to Conservative blogs (acting as cross-ideological bridges), they would appear more frequently on shortest paths between the two clusters - giving them higher betweenness scores. This does not mean Liberal blogs are more popular, only that they play a structural bridging role.

Eigenvector Centrality:  
We would expect eigenvector centrality to be high for the most prominent blogs on each side - major political commentary sites that are linked to by many other influential blogs. The top eigenvector scores may be roughly balanced between the two groups, since each side has its own hierarchy of influential nodes.

### Predicted Outcome Table

| Centrality Measure | Liberal Blogs | Conservative Blogs | Interpretation |
|---|---|---|---|
| Degree Centrality | Lower | Higher | Conservative cluster is denser |
| Betweenness Centrality | Higher | Lower | Liberal blogs bridge across groups more |
| Eigenvector Centrality | Similar | Similar | Both sides have their own influential hubs |
| In-degree Centrality | Similar | Similar | Both sides receive links primarily from within their group |

### Business / Research Implication

If this hypothesis is confirmed, it would suggest that targeting high-betweenness Liberal blogs would be the most efficient strategy for spreading information across the entire political network - reaching both sides. Targeting high-degree Conservative blogs would be more effective for reaching deep within the Conservative community specifically. This distinction between reach within a community (degree centrality) and reach across communities (betweenness centrality) is a key practical insight that centrality analysis provides over simple follower counts.

### Reference

Adamic, L. A., & Glance, N. (2005). The political blogosphere and the 2004 US election: Divided they blog. Proceedings of the 3rd International Workshop on Link Discovery, 36-43. https://dl.acm.org/doi/10.1145/1134271.1134277